# Model Evaluation
Load a trained model and evaluate prediction accuracy + game performance.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from game import GameConfig, sample_transitions, play_game, play_games_batched
from engine import Engine
from model import load_model, make_leaf_fn, make_array_leaf_fn, encode_state_arrays, greedy_nn_action

In [ ]:
# --- CONFIG: set these ---
MODEL_PATH = "models/d2_f10_24k_e50.pt"
DEPTH = 2
FANOUT = 10
N_STATES = 200    # states per game stage for prediction test
N_GAMES = 100     # games for benchmark

In [ ]:
config = GameConfig.small()
model = load_model(MODEL_PATH, config)
print(f"Model: {MODEL_PATH}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Config: {config.num_types} types, {config.num_bundles} bundles, ~{config.num_rounds} rounds/game")

# Warmup numba
Engine(1, 5, config).search_value(config.init_stashed, config.init_pool)
print("Ready.")

## Prediction vs Search Value
Sample random states at different game stages. Compare NN prediction to heuristic search value.

In [ ]:
def sample_states(config, target_round, n):
    """Play randomly to reach target_round, sample n states."""
    stashed = np.empty((n, config.num_types), dtype=np.int64)
    remaining = np.empty((n, config.num_types), dtype=np.int64)
    for i in range(n):
        s, r = config.init_stashed, config.init_pool
        for _ in range(target_round):
            if sum(r) < config.draw_size:
                break
            transitions, _, _ = sample_transitions(s, r, config)
            s, r = transitions[np.random.randint(len(transitions))]
        stashed[i] = s
        remaining[i] = r
    return stashed, remaining

rounds_to_test = [r for r in range(1, config.num_rounds - 1)]
round_data = {}

for rd in rounds_to_test:
    stashed, remaining = sample_states(config, rd, N_STATES)
    alive = remaining.sum(axis=1) >= config.draw_size
    if alive.sum() == 0:
        continue
    stashed, remaining = stashed[alive], remaining[alive]
    n = len(stashed)
    
    # NN predictions
    X = encode_state_arrays(stashed, remaining, config)
    with torch.no_grad():
        nn_preds = model(X).numpy()
    
    # Search values
    eng = Engine(DEPTH, FANOUT, config)
    search_vals = np.array([
        eng.search_value(tuple(stashed[i]), tuple(remaining[i]))
        for i in range(n)
    ])
    
    deck = int(remaining[0].sum())
    round_data[rd] = dict(nn=nn_preds, search=search_vals, n=n, deck=deck)
    
    err = nn_preds - search_vals
    rmse = np.sqrt(np.mean(err**2))
    corr = np.corrcoef(nn_preds, search_vals)[0, 1]
    print(f"Round {rd} ({n} states, deck={deck}): RMSE={rmse:.2f}, corr={corr:.4f}")

In [ ]:
fig, axes = plt.subplots(1, len(round_data), figsize=(5 * len(round_data), 4.5), squeeze=False)

for i, (rd, d) in enumerate(sorted(round_data.items())):
    ax = axes[0, i]
    ax.scatter(d['search'], d['nn'], alpha=0.4, s=10)
    
    # y=x line
    lo = min(d['search'].min(), d['nn'].min())
    hi = max(d['search'].max(), d['nn'].max())
    ax.plot([lo, hi], [lo, hi], 'r--', alpha=0.5, label='y=x')
    
    err = d['nn'] - d['search']
    rmse = np.sqrt(np.mean(err**2))
    corr = np.corrcoef(d['nn'], d['search'])[0, 1]
    
    ax.set_xlabel('Search value (heuristic)')
    ax.set_ylabel('NN prediction')
    ax.set_title(f'Round {rd} (deck={d["deck"]})\nRMSE={rmse:.1f}, r={corr:.3f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle(f'NN Prediction vs Search (d={DEPTH}, f={FANOUT})', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Error distributions
fig, axes = plt.subplots(1, len(round_data), figsize=(5 * len(round_data), 3.5), squeeze=False)

for i, (rd, d) in enumerate(sorted(round_data.items())):
    ax = axes[0, i]
    err = d['nn'] - d['search']
    ax.hist(err, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='r', linestyle='--', alpha=0.5)
    ax.axvline(err.mean(), color='orange', linestyle='-', label=f'mean={err.mean():.1f}')
    ax.set_xlabel('Prediction error (NN - search)')
    ax.set_ylabel('Count')
    ax.set_title(f'Round {rd} (deck={d["deck"]})')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Prediction Error Distribution', y=1.02)
plt.tight_layout()
plt.show()

## Game Performance Benchmark
Play full games with different strategies and compare scores.

In [ ]:
import time

results = {}

# Random
t0 = time.time()
results['Random'] = np.array([play_game(config, lambda t: np.random.randint(len(t))) for _ in range(N_GAMES)])
print(f"Random: {time.time()-t0:.1f}s")

# Heuristic (depth=0)
t0 = time.time()
eng = Engine(0, 0, config)
results['Heuristic (d=0)'] = np.array([play_game(config, eng.get_action) for _ in range(N_GAMES)])
print(f"Heuristic d=0: {time.time()-t0:.1f}s")

# NN only
t0 = time.time()
results['NN only'] = np.array([play_game(config, lambda t: greedy_nn_action(model, t, config)) for _ in range(N_GAMES)])
print(f"NN only: {time.time()-t0:.1f}s")

# Heuristic search
t0 = time.time()
eng = Engine(DEPTH, FANOUT, config)
results[f'Heuristic d={DEPTH}'] = play_games_batched(N_GAMES, eng, config)["scores"]
print(f"Heuristic d={DEPTH}: {time.time()-t0:.1f}s")

# NN leaf search
t0 = time.time()
array_leaf_fn = make_array_leaf_fn(model, config)
eng = Engine(DEPTH, FANOUT, config, array_leaf_fn=array_leaf_fn)
results[f'NN leaf d={DEPTH}'] = play_games_batched(N_GAMES, eng, config)["scores"]
print(f"NN leaf d={DEPTH}: {time.time()-t0:.1f}s")

In [ ]:
# Score comparison table
print(f"{'Strategy':<25s} {'Mean':>7s} {'Std':>7s} {'p10':>7s} {'p50':>7s} {'p90':>7s}")
print("-" * 60)
for name, scores in results.items():
    p10, p50, p90 = np.percentile(scores, [10, 50, 90])
    print(f"{name:<25s} {np.mean(scores):7.1f} {np.std(scores):7.1f} {p10:7.1f} {p50:7.1f} {p90:7.1f}")

In [ ]:
# Score distributions
fig, ax = plt.subplots(figsize=(10, 5))

positions = list(range(len(results)))
data = [scores for scores in results.values()]
bp = ax.boxplot(data, positions=positions, widths=0.6, patch_artist=True)

colors = plt.cm.Set2(np.linspace(0, 1, len(results)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)

ax.set_xticks(positions)
ax.set_xticklabels(results.keys(), rotation=15, ha='right')
ax.set_ylabel('Game Score')
ax.set_title(f'Score Distribution by Strategy (d={DEPTH}, f={FANOUT}, n={N_GAMES})')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()